# Light Demo

This demo shows the simpelst way to create mobility and charging profiles with `CHAMPPY`. Therefore, no paramerization is performed, but existing model parameters are used. 

## 1. Import Required Libraries
Import all necessary libraries, including pandas and champpy.

In [1]:
import pandas as pd
import champpy

## 2. Check available model parameters

The model champpy provides different sets of model parameters, e.g. for different vehicles types. First you have to check which parameters are available using the `ParamsLoader`. 

In [2]:
# generate instance of ParamsLoader
params_loader= champpy.ParamsLoader()

# load info DataFrame
params_info_df = params_loader.load_info()
params_info_df


,id_params,description,vehicle_type,temp_res,annual_km,locations,share_of_time_at_locations,number_typedays,number_clusters,labels_locations,labels_clusters,created_user,created_dt
0,1,Vans based on REM2030 Dataset (https://www.isi...,Van,0.25,21743.995,"[0, 1, 3]","[12.411, 58.285, 29.304]",7,1,"[Driving, Depot, Other location]",[Cluster 1],FBiedenbach,2026-03-11 12:28:44.336818
1,2,Trucks based on Nefton Dataset (https://doi.or...,Truck,0.25,59064.024,"[0, 1, 3, 5, 6]","[15.821, 54.941, 15.94, 4.322, 8.976]",7,1,"[Driving, Depot, Other location, Rest area, In...",[Cluster 1],FBiedenbach,2026-03-11 12:33:13.254130
2,3,Passenger cars based on KID 2010,Passenger car,0.25,11568.039,"[0, 1, 3]","[9.588, 18.307, 72.105]",7,1,"[Driving, Company site, Other location]",[Cluster 1],FBiedenbach,2026-03-11 12:39:39.712594


Select the `id_params` of the model parameters you want to use. In this example, we use parameters for vans.

In [3]:
selected_id_params = 1
model_params = params_loader.load_params(id_params=selected_id_params)

[2026-03-12 11:45:05 - INFO - champpy.core.mobility.parameterization] Load parameters with id_params=1.


## 3. Generate Mobility Profiles
Using the `MobModel` and `UserParamsMobModel` to generate synthetic mobility profiles for a specified number of vehicles and date range. The model parameters loaded in the previous step, serves as input to create the instance of `MobModel`.

In [4]:
# Generate synthetic mobility profiles
mob_model = champpy.MobModel(model_params=model_params)
user_params_mob = champpy.UserParamsMobModel(
    number_vehicles=50,
    start_date=pd.Timestamp("2025-01-01-00:00:00"),
    end_date=pd.Timestamp("2025-12-31-23:00:00"),
)
mob_profiles = mob_model.generate_mob_profiles(user_params=user_params_mob)

[2026-03-12 11:46:41 - INFO - champpy.core.mobility.mobility_model] Start generating mobility profiles for 50 vehicles from 2025-01-01 00:00:00 to 2025-12-31 23:00:00


Output()

Display logbook of the generated mobility profile:

In [5]:
mob_profiles.logbooks.df.head()

,id_journey,id_vehicle,dep_dt,arr_dt,dep_loc,arr_loc,distance,duration,speed
0,1,1,2025-01-01 07:00:00,2025-01-01 08:45:00,1,3,40.738056,1.75,23.278889
1,2,1,2025-01-01 10:30:00,2025-01-01 11:15:00,3,1,15.549082,0.75,20.732109
2,3,1,2025-01-01 11:30:00,2025-01-01 12:00:00,1,1,1.205889,0.50,2.411779
3,4,1,2025-01-02 07:30:00,2025-01-02 07:45:00,1,1,1.420597,0.25,5.682388
4,5,1,2025-01-02 09:15:00,2025-01-02 09:45:00,1,1,5.628537,0.50,11.257073


## 4. Generate charging profiles
Use classes`ChargeModel` and `UserParamsChargeModel` to generate synthetic charging profiles.

In [6]:
# Initilaize the charging model with the modeled mobility prfiles
charging_model = champpy.ChargingModel(mob_profiles)

# Define user parameters for the charging model
user_params_charging = champpy.UserParamsChargingModel(
    energy_consumption_kwh_per_km=[0.2],
    battery_capacity_kwh=[80.0],
    charging_power_max_kw=[11],
    efficiency_charging=[0.9],
    soc_min=[0.1],
    soc_min_dep=[0.8],
    soc_initial=1,
    distribute_energy_consumption=True,
    charging_locations=[1], 
    temp_res=0.25
)

# Generate charging profiles based on the mobility profiles and the user parameters for charging
charging_profiles = charging_model.generate_charging_profiles(user_params=user_params_charging)

[2026-03-12 11:46:48 - INFO - champpy.core.mobility.mobility_data] Creating MobArray from MobProfiles
[2026-03-12 11:46:48 - INFO - champpy.core.mobility.mobility_data] Extending MobProfiles
[2026-03-12 11:46:50 - INFO - root] Generating charging profiles based on mobility data and user parameters...


Output()

Display timeseries of the generated charging profiles:

In [8]:
charging_profiles.charging_timeseries.df.head()

,id_vehicle,datetime,connected,energy_consumption_kwh,energy_stored_kwh,power_charging_kw,energy_missing_kwh
0,1,2025-01-01 00:00:00,False,0.0,50.0,0.0,0.0
1,1,2025-01-01 00:15:00,False,0.0,50.0,0.0,0.0
2,1,2025-01-01 00:30:00,False,0.0,50.0,0.0,0.0
3,1,2025-01-01 00:45:00,False,0.0,50.0,0.0,0.0
4,1,2025-01-01 01:00:00,False,0.0,50.0,0.0,0.0


## 5. Plot mobility and charging profiles

You can visulize the modeld mobility and charging profiles using the `MobPlotter` and `ChargingPlotter` classed. They must be initialized by different user parameters defined in the `UserParamsMobPlotter` and `UserParamsChargingPlotter` data classes. 

In [ ]:
# Initialize user parameters for plotting the mobiltiy profiles
user_params_plot = champpy.UserParamsMobPlotter(
    filename="demo01_mobility_profiles_plot.html",
    clustering = False
)

# Create instance of the mobility plotter
mobplot = champpy.MobPlotter(user_params_plot)

# Plot the mobility profiles for the merged data (ref + model)
mobplot.plot_mob_profiles(mob_profiles)

# Initialize user parameters for plotting the charging profiles
user_params_plot = champpy.UserParamsChargingPlotter(
    filename="demo01_charging_profiles_plot.html",
    clustering = False
)

# Create an instance of the ChargingPlotter
chargeplot = champpy.ChargingPlotter(user_params_plot)

# Plot the charging profiles
chargeplot.plot_charging_profiles(charging_profiles)